# Data preprocessing

### Analysons d'abord les donnees presents

In [82]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

import warnings
warnings.filterwarnings('ignore')

In [83]:
data = pd.read_csv('evmar.csv')

In [84]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1930 entries, 0 to 1929
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Date debut         1862 non-null   object
 1   Date fin           22 non-null     object
 2   Thématique         1862 non-null   object
 3   Objets             1617 non-null   object
 4   District           1711 non-null   object
 5   Commune            100 non-null    object
 6   Localite           871 non-null    object
 7   Region             28 non-null     object
 8   Types              194 non-null    object
 9   Longitude          1513 non-null   object
 10  Latitude           1513 non-null   object
 11  Personne concerne  7 non-null      object
 12  Mort               4 non-null      object
 13  Colonne1           14 non-null     object
 14  description        1813 non-null   object
 15  Commentaire        503 non-null    object
dtypes: object(16)
memory usage: 241.4+ KB


In [85]:
data["Date debut"].unique()

array(['9/22/2017', '12/27/2017', nan, '2/10/2017', '2/15/2017',
       '3/10/2017', '3/16/2017', '2/11/2017', '2/25/2017', '3/17/2017',
       '3/31/2017', '4/8/2017', '4/17/2017', '4/20/2017', '5/2/2017',
       '6/8/2017', '7/3/2017', '7/8/2017', '7/25/2017', '7/29/2017',
       '8/12/2017', '9/5/2017', '9/7/2017', '9/13/2017', '9/21/2017',
       '9/24/2017', '9/28/2017', '10/8/2017', '10/13/2017', '10/30/2017',
       '11/7/2017', '11/8/2017', '11/11/2017', '11/16/2017', '11/27/2017',
       '12/13/2017', '12/25/2017', '1/25/2017', '10/10/2017',
       '10/27/2017', '12/16/2017', '4/18/2017', '6/30/2017', '9/2/2017',
       '11/9/2017', '5/18/2017', '2/6/2017', '1/10/2018', '2/15/2018',
       '3/8/2018', '3/27/2018', '4/6/2018', '5/10/2018', '5/12/2018',
       '5/15/2018', '5/19/2018', '6/18/2018', '7/9/2018', '7/24/2018',
       '8/31/2018', '9/6/2018', '9/12/2018', '9/13/2018', '10/20/2018',
       '11/8/2018', '12/9/2018', '3/10/2018', '2/2/2018', '2/23/2018',
       '2/27/20

In [86]:
data["Date fin"].unique()

array([nan, '4/10/2019', '7/6/2019', '27/9/2019', '15/10/2019',
       '21/10/2019', '25/10/2019', '26/10/2019', '12/9/2019',
       '11/17/2019', '11/18/2019', '11/24/2019', '11/30/2019',
       '12/6/2019', '12/10/2019', '12/11/2019', '8/30/2019', '20/9/2019',
       '29/9/2019', '20/10/2019'], dtype=object)

In [87]:
data["Thématique"].unique()

array(['Autres', nan, 'Evènement naturel',
       'Evènement naturel maritime/ AHSC',
       'Evènement naturel maritime/ AHSC ', 'Incidents maritimes',
       'Incidents Maritimes', 'Incidents Maritimes ',
       'Infrastructure critique',
       'Pêche Illégale, Non Reportée et Non Règlementée (INN)',
       'Trafic d’armes, de drogues et contrebande',
       'Trafic d’armes, de drogues et contrebande ',
       'Trafics et contrebandes', 'VAPM',
       'Migration illégale & Trafic d’être humain par voie maritime',
       'Trafic et contrebande par voie maritime', 'Environnement Marin',
       'Evenement naturel maritime', 'Incident maritime',
       'Infrastructure critique maritime',
       "Migration irrégulière et trafics d'être humain par voie maritime et trafic d’être humain par voie maritime",
       "Migration irrégulière et trafics d'être humain par voie maritime et trafic d’être humain par voie maritime ",
       'Migration illégale & trafic d’être humain par voie maritime',

In [88]:
data["Longitude"].unique()

array(['44.05', '47.008206', nan, ..., '48.215488', '46.311833',
       '46.308367'], shape=(1300,), dtype=object)

In [89]:
data["Latitude"].unique()

array(['-27.1', '-12.865347', nan, ..., '-13.386639', '-15.727361',
       '-15.702215'], shape=(1294,), dtype=object)

In [90]:
data["description"].unique()

array(['Le Commandant russe du navire MT ETC MENA, pavillon Libéria, a été poignardé par son 3ème Officier, de nationalité philippine lors de son transit dans l’Océan indien, Ce dernier s’est jeté par-dessus bord après son acte, Selon les informations, le Commandant a succombé à ses blessures, Les moyens de communication du navire restent indisponibles car enfermés dans la cabine du Commandant, dernière position transmise dudit navire : 27° 06 S / 044°03 E, 101 Nautiques au SW d’Ambovombe,  ',
       'A la position 12°52S et 047°E, le FV GIBELLE, pavillon Espagnol, a reporté qu’au cours de son transit, il a été poursuivi et attaqué par un dhow et 2 skiffs avec 4 personnes à bord. Une équipe de protection armée était également à bord.',
       nan, ...,
       'Dans la nuit 24 octobre 2022 les éléments du Détachement de la Marine Nationale de Nosy Be ont intercepté un boutre à voile malagasy nommé TOLIAMEVA au large du quartier de Diégo Hely dans fokontany Ambatolaoka dans la CU Dzamand

Apres avoir analyser les donnees voila ce qu'on va faire :

1. **Standardisation des dates** : 
   - Uniformiser toutes les dates au format `dd/mm/yyyy`
   - Gérer les différents formats de dates présents dans le dataset

2. **Filtrage temporel** : 
   - Conserver uniquement les données de la période 2017-2022
   - Trier les données par ordre chronologique (date de début)
   
3. **Filtrage thématique** : 
   - Sélectionner uniquement les données relatives aux incidents maritimes
   - Nettoyer les variations dans les libellés de thématiques


4. **Completer les donnees**:
   - On complete les donnees pour les dates debuts manquantes entre 1/1/2017 a 31/12/2022 avec la description : 'Aucun incident maritime'

5. **Uniformiser les donnees dans longitude et latitude en decimale**:

6. **Validation et nettoyage** :
   - Vérifier la cohérence des données filtrées
   - Supprimer les doublons éventuels

### 1. Standardisation des dates

In [91]:
def standardize_date(date_str):
    """
    Standardise les dates au format dd/mm/yyyy
    Gère plusieurs formats d'entrée et retourne None pour les dates invalides
    """
    if pd.isna(date_str):
        return None

    date_str = str(date_str).strip()
    
    if not date_str or date_str.lower() in ['nan', 'none', 'null', '']:
        return None

    # Liste des formats possibles à tester
    formats = [
        '%d/%m/%Y',    # 01/12/2020
        '%d/%m/%y',    # 01/12/20
        '%Y-%m-%d',    # 2020-12-01
        '%d-%m-%Y',    # 01-12-2020
        '%d-%m-%y',    # 01-12-20
        '%d.%m.%Y',    # 01.12.2020
        '%d.%m.%y',    # 01.12.20
        '%Y/%m/%d',    # 2020/12/01
    ]

    for fmt in formats:
        try:
            date_obj = pd.to_datetime(date_str, format=fmt, errors='raise')
            # Retourner au format dd/mm/yyyy
            return date_obj.strftime('%d/%m/%Y')
        except:
            continue

    # Essai avec parser automatique en dernier recours
    try:
        date_obj = pd.to_datetime(date_str, errors='coerce', dayfirst=True)
        if pd.notna(date_obj):
            return date_obj.strftime('%d/%m/%Y')
    except:
        pass

    # Si aucun format ne fonctionne, retourner None
    print(f"Date non reconnue: {date_str}")
    return None

Application de la standarisation des dates aux date debut et fin

In [92]:
date_debut_standardized = data['Date debut'].apply(standardize_date)
data['Date debut'] = date_debut_standardized

In [93]:
date_fin_standardized = data['Date fin'].apply(standardize_date)
data['Date fin'] = date_fin_standardized

Affichage des dates standarises

In [94]:
data["Date debut"].unique()

array(['22/09/2017', '27/12/2017', None, '02/10/2017', '15/02/2017',
       '03/10/2017', '16/03/2017', '02/11/2017', '25/02/2017',
       '17/03/2017', '31/03/2017', '04/08/2017', '17/04/2017',
       '20/04/2017', '05/02/2017', '06/08/2017', '07/03/2017',
       '07/08/2017', '25/07/2017', '29/07/2017', '08/12/2017',
       '09/05/2017', '09/07/2017', '13/09/2017', '21/09/2017',
       '24/09/2017', '28/09/2017', '10/08/2017', '13/10/2017',
       '30/10/2017', '11/07/2017', '11/08/2017', '11/11/2017',
       '16/11/2017', '27/11/2017', '13/12/2017', '25/12/2017',
       '25/01/2017', '10/10/2017', '27/10/2017', '16/12/2017',
       '18/04/2017', '30/06/2017', '09/02/2017', '11/09/2017',
       '18/05/2017', '02/06/2017', '01/10/2018', '15/02/2018',
       '03/08/2018', '27/03/2018', '04/06/2018', '05/10/2018',
       '05/12/2018', '15/05/2018', '19/05/2018', '18/06/2018',
       '07/09/2018', '24/07/2018', '31/08/2018', '09/06/2018',
       '09/12/2018', '13/09/2018', '20/10/2018', 

### 2. Filtrage temporel

Convertion des date debut

In [95]:
if len(data) == 0:
    print("Aucune donnee")
else:
    print("Conversion des dates de début...")
    data['Date_debut_dt'] = pd.to_datetime(
        data['Date debut'], 
        format='%d/%m/%Y', 
        errors='coerce'
    )

Conversion des dates de début...


Statistiques des dates

In [96]:
dates_valides = data['Date_debut_dt'].notna().sum()
dates_invalides = data['Date_debut_dt'].isna().sum()
    
print(f"Dates valides: {dates_valides}")
print(f"Dates invalides: {dates_invalides}")
    
if dates_valides > 0:
    print(f"Période des données: {data['Date_debut_dt'].min().strftime('%d/%m/%Y')} à {data['Date_debut_dt'].max().strftime('%d/%m/%Y')}")

Dates valides: 1862
Dates invalides: 68
Période des données: 25/01/2017 à 30/08/2023


Définition de la période cible

In [97]:
start_date = pd.to_datetime('01/01/2017', format='%d/%m/%Y')
end_date = pd.to_datetime('31/12/2022', format='%d/%m/%Y')

print(f"\nPériode de filtrage: {start_date.strftime('%d/%m/%Y')} à {end_date.strftime('%d/%m/%Y')}")


Période de filtrage: 01/01/2017 à 31/12/2022


Application du filtre temporel

In [98]:
mask_periode = (
    (data['Date_debut_dt'] >= start_date) & 
    (data['Date_debut_dt'] <= end_date)
)

In [99]:
data_filtered = data[mask_periode].copy()
    
    # Tri par date croissante
data_sorted = data_filtered.sort_values(
    by='Date_debut_dt', 
    ascending=True
).reset_index(drop=True)

# Suppression de la colonne temporaire
data_sorted = data_sorted.drop('Date_debut_dt', axis=1)

### 3. Filtrage thematique

In [100]:
def filter_maritime_incidents(df):
    """
    Filtre les données pour ne conserver que les incidents maritimes
    Utilise plusieurs critères de recherche pour capturer toutes les variantes
    """
    print("=== FILTRAGE DES INCIDENTS MARITIMES ===")
    
    # Normalisation de la colonne thématique
    df['Thematique_norm'] = df['Thématique'].str.lower().str.strip()
    
    # Affichage des thématiques uniques pour diagnostic
    print("Thématiques présentes dans les données:")
    thematiques_uniques = df['Thematique_norm'].dropna().unique()
    for i, theme in enumerate(sorted(thematiques_uniques), 1):
        print(f"  {i}. {theme}")
    
    # Critères de recherche pour les incidents maritimes
    patterns_maritime = [
        'incident.*maritime',
        'maritime.*incident',
        'accident.*maritime', 
        'maritime.*accident',
        'incident.*mer',
        'mer.*incident',
        'navire',
        'bateau',
        'embarcation'
    ]
    
    # Création du masque de filtrage
    mask = pd.Series([False] * len(df))
    
    for pattern in patterns_maritime:
        pattern_mask = df['Thematique_norm'].str.contains(pattern, na=False, regex=True)
        mask = mask | pattern_mask
        matches = pattern_mask.sum()
        if matches > 0:
            print(f"Pattern '{pattern}': {matches} correspondances")
    
    # Application du filtre
    maritime_incidents = df[mask].copy()
    
    print(f"\nRésultat du filtrage:")
    print(f"  - Données originales: {len(df)} lignes")
    print(f"  - Incidents maritimes: {len(maritime_incidents)} lignes")
    print(f"  - Pourcentage conservé: {len(maritime_incidents)/len(df)*100:.1f}%")
    
    # Nettoyage de la colonne temporaire
    maritime_incidents = maritime_incidents.drop('Thematique_norm', axis=1)
    
    return maritime_incidents

In [101]:
incident_data = filter_maritime_incidents(data)

=== FILTRAGE DES INCIDENTS MARITIMES ===
Thématiques présentes dans les données:
  1. acte violent en mer
  2. actes violents en mer
  3. attaque violent en mer (vam)
  4. autres
  5. environnement marin
  6. evenement naturel maritime
  7. evènement naturel
  8. evènement naturel maritime
  9. evènement naturel maritime (ahsc)
  10. evènement naturel maritime/ ahsc
  11. evénement naturel maritime
  12. illegal, unreported & unregulated fishing (iuu)
  13. incident maritime
  14. incidents maritimes
  15. infrastructure critique
  16. infrastructure critique maritime
  17. infrastructures
  18. irregular migration and illicit trafficking/of migrants by sea
  19. migration illégale & trafic d’être humain par voie maritime
  20. migration irrégulière et trafics d'être humain par voie maritime et trafic d’être humain par voie maritime
  21. plaisance et tourisme maritime
  22. plaisance/tourisme maritime
  23. pêche illégale non reportée et non règlementée (inn)
  24. pêche illégale non 

### 4. Completer les donnees

In [102]:
def completer_dates_manquantes(df):
    """
    Complète les dates manquantes entre 2017 et 2022 avec 'Aucun incident maritime'
    """
    print("=== COMPLETION DES DATES MANQUANTES ===")
    
    if len(df) == 0:
        print("Aucune donnée à traiter")
        return df
    
    # Créer une liste complète de toutes les dates de 2017 à 2022
    start_date = pd.to_datetime('01/01/2017', format='%d/%m/%Y')
    end_date = pd.to_datetime('31/12/2022', format='%d/%m/%Y')
    
    print(f"Période de completion: {start_date.strftime('%d/%m/%Y')} à {end_date.strftime('%d/%m/%Y')}")
    
    # Générer toutes les dates dans la période
    all_dates = pd.date_range(start=start_date, end=end_date, freq='D')
    all_dates_str = [date.strftime('%d/%m/%Y') for date in all_dates]
    
    print(f"Nombre total de jours dans la période: {len(all_dates_str)}")
    
    # Convertir les dates de début existantes en format datetime pour comparaison
    df_dates = pd.to_datetime(df['Date debut'], format='%d/%m/%Y', errors='coerce')
    existing_dates = set(df_dates.dropna().dt.strftime('%d/%m/%Y'))
    
    print(f"Nombre de dates existantes avec incidents: {len(existing_dates)}")
    
    # Identifier les dates manquantes
    missing_dates = set(all_dates_str) - existing_dates
    missing_dates = sorted(list(missing_dates))
    
    print(f"Nombre de dates manquantes: {len(missing_dates)}")
    
    # Créer des lignes pour les dates manquantes
    if len(missing_dates) > 0:
        # Obtenir la structure des colonnes du dataframe existant
        columns = df.columns.tolist()
        
        # Créer le DataFrame pour les dates manquantes
        missing_data = []
        
        for date_str in missing_dates:
            row_data = {}
            for col in columns:
                if col == 'Date debut':
                    row_data[col] = date_str
                elif col == 'Date fin':
                    row_data[col] = date_str  # Même date pour début et fin
                elif col == 'description':
                    row_data[col] = 'Aucun incident maritime'
                elif col == 'Thématique':
                    row_data[col] = 'Incident maritime'  # Pour maintenir la cohérence
                else:
                    row_data[col] = None  # Valeur vide pour les autres colonnes
            
            missing_data.append(row_data)
        
        # Créer le DataFrame des dates manquantes
        missing_df = pd.DataFrame(missing_data)
        
        # Combiner avec les données existantes
        completed_df = pd.concat([df, missing_df], ignore_index=True)
        
        # Trier par date
        completed_df['Date_temp'] = pd.to_datetime(completed_df['Date debut'], format='%d/%m/%Y')
        completed_df = completed_df.sort_values('Date_temp').drop('Date_temp', axis=1).reset_index(drop=True)
        
        print(f"\nRésultat:")
        print(f"  - Données originales: {len(df)} lignes")
        print(f"  - Dates ajoutées: {len(missing_data)} lignes")
        print(f"  - Total final: {len(completed_df)} lignes")
        
        # Vérification de quelques exemples
        aucun_incident_count = (completed_df['description'] == 'Aucun incident maritime').sum()
        incidents_reels_count = (completed_df['description'] != 'Aucun incident maritime').sum()
        
        print(f"\nVérification:")
        print(f"  - Jours sans incident: {aucun_incident_count}")
        print(f"  - Jours avec incidents: {incidents_reels_count}")
        
        return completed_df
    
    else:
        print("Toutes les dates sont déjà présentes!")
        return df

In [103]:
# Application de la completion des dates manquantes
incident_data_complete = completer_dates_manquantes(incident_data)

print("✓ Completion des dates terminée")

=== COMPLETION DES DATES MANQUANTES ===
Période de completion: 01/01/2017 à 31/12/2022
Nombre total de jours dans la période: 2191
Nombre de dates existantes avec incidents: 245
Nombre de dates manquantes: 1946

Résultat:
  - Données originales: 269 lignes
  - Dates ajoutées: 1946 lignes
  - Total final: 2215 lignes

Vérification:
  - Jours sans incident: 1946
  - Jours avec incidents: 269
✓ Completion des dates terminée


### 5. Uniformiser les longitudes et latitudes

In [105]:
longitudes = incident_data_complete["Longitude"].unique()
latitudes = incident_data_complete["Latitude"].unique()

In [106]:
def dms_to_decimal(dms_str):
    """
    Convertit une coordonnée au format DMS (degrés, minutes, secondes) en décimal.
    Exemple: '49°51\'12.59"E' -> 49.853497
    """
    dms_str = dms_str.replace(" ", "")
    match = re.match(r'(\d+)°(\d+)?\'?([\d\.]+)?"?([NSEW]?)', dms_str)
    if not match:
        return np.nan
    deg, minute, sec, direction = match.groups()
    deg = float(deg)
    minute = float(minute) if minute else 0
    sec = float(sec) if sec else 0
    decimal = deg + minute/60 + sec/3600
    if direction in ['W', 'S']:
        decimal = -decimal
    return decimal

def normalize_longitude(val):
    if pd.isna(val) or val in ["None", "nan", "NaN", ""]:
        return np.nan
    val = str(val).strip()
    
    # Cas DMS
    if "°" in val and ("'" in val or '"' in val):
        return dms_to_decimal(val)
    
    # Cas décimal avec symbole °
    if "°" in val:
        try:
            return float(val.replace("°", ""))
        except:
            return np.nan
    
    # Cas normal float
    try:
        return float(val)
    except:
        return np.nan
   

In [107]:
longitudes_decimal = [normalize_longitude(lon) for lon in longitudes]
latitudes_decimal = [normalize_longitude(lat) for lat in latitudes]

In [108]:
incident_data_complete["Longitude"] = incident_data_complete["Longitude"].apply(normalize_longitude)
incident_data_complete["Latitude"] = incident_data_complete["Latitude"].apply(normalize_longitude)

### 6. Validation et nettoyage

In [109]:
def validation_et_nettoyage(df):
    """
    Valide et nettoie les données finales
    - Vérifie la cohérence des données
    - Supprime les doublons
    - Affiche un résumé final
    """
    print("=== VALIDATION ET NETTOYAGE FINAL ===")
    
    if len(df) == 0:
        print("Aucune donnée à valider")
        return df
    
    print(f"Données avant nettoyage: {len(df)} lignes")
    
    # 1. Vérification des dates
    print(f"\n1. Vérification des dates...")
    dates_invalides = pd.to_datetime(df['Date debut'], format='%d/%m/%Y', errors='coerce').isna().sum()
    print(f"   - Dates invalides: {dates_invalides}")
    
    # 2. Vérification de la période
    df_temp = df.copy()
    df_temp['Date_dt'] = pd.to_datetime(df_temp['Date debut'], format='%d/%m/%Y', errors='coerce')
    start_check = pd.to_datetime('01/01/2017')
    end_check = pd.to_datetime('31/12/2022')
    
    hors_periode = ((df_temp['Date_dt'] < start_check) | (df_temp['Date_dt'] > end_check)).sum()
    print(f"   - Dates hors période 2017-2022: {hors_periode}")
    
    # 3. Détection des doublons
    print(f"\n2. Détection des doublons...")
    
    # Doublons exacts (toutes colonnes identiques)
    doublons_exacts = df.duplicated().sum()
    print(f"   - Doublons exacts (toutes colonnes): {doublons_exacts}")
    
    # Doublons par date et description (plus probable)
    doublons_date_desc = df.duplicated(subset=['Date debut', 'description']).sum()
    print(f"   - Doublons par date + description: {doublons_date_desc}")
    
    # 4. Suppression des doublons
    print(f"\n3. Suppression des doublons...")
    df_cleaned = df.drop_duplicates().reset_index(drop=True)
    doublons_supprimes = len(df) - len(df_cleaned)
    print(f"   - Doublons supprimés: {doublons_supprimes}")
    
    # 5. Vérifications finales
    print(f"\n4. Statistiques finales...")
    print(f"   - Lignes finales: {len(df_cleaned)}")
    
    # Distribution par type d'incident
    incidents_reels = (df_cleaned['description'] != 'Aucun incident maritime').sum()
    jours_sans_incident = (df_cleaned['description'] == 'Aucun incident maritime').sum()
    
    print(f"   - Jours avec incidents réels: {incidents_reels}")
    print(f"   - Jours sans incident: {jours_sans_incident}")
    print(f"   - Total: {incidents_reels + jours_sans_incident}")
    
    # Vérification de la continuité des dates
    df_cleaned_temp = df_cleaned.copy()
    df_cleaned_temp['Date_dt'] = pd.to_datetime(df_cleaned_temp['Date debut'], format='%d/%m/%Y')
    df_cleaned_temp = df_cleaned_temp.sort_values('Date_dt')
    
    date_min = df_cleaned_temp['Date_dt'].min()
    date_max = df_cleaned_temp['Date_dt'].max()
    jours_attendus = (date_max - date_min).days + 1
    
    print(f"\n5. Vérification de la continuité...")
    print(f"   - Première date: {date_min.strftime('%d/%m/%Y')}")
    print(f"   - Dernière date: {date_max.strftime('%d/%m/%Y')}")
    print(f"   - Jours attendus: {jours_attendus}")
    print(f"   - Jours présents: {len(df_cleaned)}")
    print(f"   - Continuité: {'✓ OK' if len(df_cleaned) == jours_attendus else '✗ Manquant'}")
    
    # Nettoyage de la colonne temporaire
    df_cleaned = df_cleaned.drop('Date_dt', axis=1, errors='ignore')
    
    return df_cleaned

In [110]:
# Application de la validation et nettoyage
incident_data_final = validation_et_nettoyage(incident_data_complete)

print("✓ Validation et nettoyage terminés")

=== VALIDATION ET NETTOYAGE FINAL ===
Données avant nettoyage: 2215 lignes

1. Vérification des dates...
   - Dates invalides: 0
   - Dates hors période 2017-2022: 0

2. Détection des doublons...
   - Doublons exacts (toutes colonnes): 0
   - Doublons par date + description: 1

3. Suppression des doublons...
   - Doublons supprimés: 0

4. Statistiques finales...
   - Lignes finales: 2215
   - Jours avec incidents réels: 269
   - Jours sans incident: 1946
   - Total: 2215

5. Vérification de la continuité...
   - Première date: 01/01/2017
   - Dernière date: 31/12/2022
   - Jours attendus: 2191
   - Jours présents: 2215
   - Continuité: ✗ Manquant
✓ Validation et nettoyage terminés


## Sauvegarde final des donnees

In [111]:
print("=== SAUVEGARDE FINALE ET RÉSUMÉ COMPLET ===")

if 'incident_data_final' in locals() and len(incident_data_final) > 0:
    # Sauvegarde en CSV
    output_filename_final = 'incidents_maritimes_complets_2017_2022.csv'
    incident_data_final.to_csv(output_filename_final, index=False, encoding='utf-8')
    print(f"✓ Données finales sauvegardées dans: {output_filename_final}")
    
    # Résumé complet du preprocessing
    print(f"\n{'='*60}")
    print(f"RÉSUMÉ COMPLET DU PREPROCESSING")
    print(f"{'='*60}")
    
    print(f"\n STATISTIQUES GÉNÉRALES:")
    print(f"  • Dataset original: {len(data) if 'data' in locals() else 'N/A'} lignes")
    print(f"  • Après filtrage incidents maritimes: {len(incident_data) if 'incident_data' in locals() else 'N/A'} lignes")
    print(f"  • Après filtrage période 2017-2022: {len(incident_data_sorted) if 'incident_data_sorted' in locals() else 'N/A'} lignes")
    print(f"  • Après completion des dates: {len(incident_data_complete) if 'incident_data_complete' in locals() else 'N/A'} lignes")
    print(f"  • Après validation/nettoyage: {len(incident_data_final)} lignes")
    
    # Analyse des incidents
    incidents_reels = (incident_data_final['description'] != 'Aucun incident maritime').sum()
    jours_sans_incident = (incident_data_final['description'] == 'Aucun incident maritime').sum()
    
    print(f"\n ANALYSE DES INCIDENTS:")
    print(f"  • Jours avec incidents réels: {incidents_reels}")
    print(f"  • Jours sans incident: {jours_sans_incident}")
    print(f"  • Taux d'incidents: {incidents_reels/len(incident_data_final)*100:.2f}%")
    print(f"  • Période totale couverte: 6 ans (2017-2022)")
    
    # Analyse par année
    df_temp = incident_data_final.copy()
    df_temp['Date_dt'] = pd.to_datetime(df_temp['Date debut'], format='%d/%m/%Y')
    df_temp['Annee'] = df_temp['Date_dt'].dt.year
    
    # Incidents réels par année
    incidents_par_annee = df_temp[df_temp['description'] != 'Aucun incident maritime'].groupby('Annee').size()
    
    print(f"\n DISTRIBUTION PAR ANNÉE:")
    for annee in range(2017, 2023):
        count = incidents_par_annee.get(annee, 0)
        print(f"  • {annee}: {count} incidents")
    
    # Vérification de la qualité des données
    print(f"\n QUALITÉ DES DONNÉES:")
    colonnes_completes = 0
    for col in incident_data_final.columns:
        non_null = incident_data_final[col].notna().sum()
        completude = non_null / len(incident_data_final) * 100
        print(f"  • {col}: {completude:.1f}% complète ({non_null}/{len(incident_data_final)})")
        if completude == 100:
            colonnes_completes += 1
    
    print(f"\n FICHIERS GÉNÉRÉS:")
    print(f"  • {output_filename_final} (dataset final complet)")
    if 'output_filename' in locals():
        print(f"  • {output_filename} (incidents uniquement, sans completion)")
    
    print(f"\n OBJECTIFS ATTEINTS:")
    print(f"  ✓ 1. Standardisation des dates au format dd/mm/yyyy")
    print(f"  ✓ 2. Filtrage des incidents maritimes uniquement")
    print(f"  ✓ 3. Filtrage temporel sur la période 2017-2022")
    print(f"  ✓ 4. Completion des dates manquantes avec 'Aucun incident maritime'")
    print(f"  ✓ 5. Validation et suppression des doublons")
    
    # Aperçu des données finales
    print(f"\n APERÇU DES DONNÉES FINALES:")
    display(incident_data_final.head(5))
    
    print(f"\n{'='*60}")
    print(f" PREPROCESSING TERMINÉ AVEC SUCCÈS !")
    print(f"{'='*60}")

else:
    print(" Erreur: Aucune donnée finale à sauvegarder")

=== SAUVEGARDE FINALE ET RÉSUMÉ COMPLET ===
✓ Données finales sauvegardées dans: incidents_maritimes_complets_2017_2022.csv

RÉSUMÉ COMPLET DU PREPROCESSING

 STATISTIQUES GÉNÉRALES:
  • Dataset original: 1930 lignes
  • Après filtrage incidents maritimes: 269 lignes
  • Après filtrage période 2017-2022: N/A lignes
  • Après completion des dates: 2215 lignes
  • Après validation/nettoyage: 2215 lignes

 ANALYSE DES INCIDENTS:
  • Jours avec incidents réels: 269
  • Jours sans incident: 1946
  • Taux d'incidents: 12.14%
  • Période totale couverte: 6 ans (2017-2022)

 DISTRIBUTION PAR ANNÉE:
  • 2017: 33 incidents
  • 2018: 20 incidents
  • 2019: 41 incidents
  • 2020: 49 incidents
  • 2021: 63 incidents
  • 2022: 63 incidents

 QUALITÉ DES DONNÉES:
  • Date debut: 100.0% complète (2215/2215)
  • Date fin: 87.9% complète (1946/2215)
  • Thématique: 100.0% complète (2215/2215)
  • Objets: 8.3% complète (184/2215)
  • District: 9.5% complète (210/2215)
  • Commune: 1.8% complète (39/2215)

,Date debut,Date fin,Thématique,Objets,District,Commune,Localite,Region,Types,Longitude,Latitude,Personne concerne,Mort,Colonne1,description,Commentaire,Date_debut_dt
0,01/01/2017,01/01/2017,Incident maritime,None,None,None,None,None,None,NaN,NaN,None,None,None,Aucun incident maritime,None,NaT
1,02/01/2017,02/01/2017,Incident maritime,None,None,None,None,None,None,NaN,NaN,None,None,None,Aucun incident maritime,None,NaT
2,03/01/2017,03/01/2017,Incident maritime,None,None,None,None,None,None,NaN,NaN,None,None,None,Aucun incident maritime,None,NaT
3,04/01/2017,04/01/2017,Incident maritime,None,None,None,None,None,None,NaN,NaN,None,None,None,Aucun incident maritime,None,NaT
4,05/01/2017,05/01/2017,Incident maritime,None,None,None,None,None,None,NaN,NaN,None,None,None,Aucun incident maritime,None,NaT



 PREPROCESSING TERMINÉ AVEC SUCCÈS !
